In [ ]:
import json
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict, Counter
from typing import Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup,
    set_seed
)
from peft import (
    get_peft_model,
    LoraConfig,
    TaskType,
    prepare_model_for_kbit_training
)
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score
)

# Set random seeds for reproducibility
set_seed(42)


# Check GPU availability    print(f"Memory Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')    print(f"GPU: {torch.cuda.get_device_name(0)}")

print(f"Using device: {device}")if torch.cuda.is_available():

In [ ]:
# Install required packages (including PEFT for LoRA)
!pip install -q transformers datasets accelerate scikit-learn matplotlib seaborn peft

## 1. Environment Setup and Dependencies

# Hierarchical Text Classification for Research Papers
## Fine-tuning with 7-Level Taxonomy (Kaggle Edition)

This notebook implements **true hierarchical classification** where predictions follow your taxonomy structure level-by-level:

### 🎯 How It Works:

**Traditional Approach** (❌ Can produce invalid paths):
- All 7 levels predict independently
- May predict "Natural Science > Computer Science" even if Computer Science isn't a child of Natural Science

**This Approach** (✓ Guarantees valid paths):
1. **Level 0**: Predict top-level category (e.g., "Natural Science")
2. **Level 1**: Predict ONLY among children of "Natural Science" (e.g., "Physics", "Chemistry", "Mathematics")
3. **Level 2**: Predict ONLY among children of the Level 1 prediction
4. **Continue for all 7 levels...**

Result: **100% of predictions are valid taxonomy paths!**

### 🔑 Key Features:

- **7-level taxonomy** with automatic padding for shorter paths
- **Parent-child constraints** ensuring valid taxonomy paths
- **Dual-mode architecture**: Independent training, hierarchical inference
- **Checkpoint-based training** for session recovery on Kaggle GPU
- **Comprehensive evaluation** with hierarchical metrics
- **LoRA fine-tuning** for parameter efficiency
- **Visualization** of training progress and results
- **Downloadable model package** ready after training

### 📊 What You'll Get:

- Model that ALWAYS predicts valid taxonomy paths
- Level-by-level performance metrics
- Hierarchical accuracy (correct predictions up to depth k)
- Path consistency verification (should be 100%)
- Ready-to-deploy model with all necessary artifacts
- **ZIP file for download** from Kaggle Output tab

### 🚀 Quick Start (Kaggle):

1. **Add your dataset** to this notebook using the "Add Data" button
2. Update `DATASET_NAME` variable in cell 6 to match your dataset folder name
3. Ensure your dataset contains:
   - `preprocessed_taxonomy.json`
   - `train_data/` folder with JSON files
   - `test_data/` folder with JSON files
4. Run all cells sequentially
5. After training, download the model ZIP from the **Output** tab

### 💾 Kaggle Output:

All model files will be saved to the Kaggle Output tab for easy download:
- Model weights
- Tokenizer
- Label encoders
- Parent-child mappings
- Training metrics
- README with usage instructions

## 2. Kaggle Data Setup

In [ ]:
# Kaggle Data Setup
# In Kaggle, your datasets are automatically mounted in /kaggle/input/

import os
import sys

# Check if running on Kaggle
if os.path.exists('/kaggle/input/'):
    print("✓ Running on Kaggle")
    
    # List available datasets
    print("\nAvailable datasets in /kaggle/input/:")
    try:
        datasets = os.listdir('/kaggle/input/')
        for dataset in datasets:
            print(f"  - {dataset}")
        
        # Set the data path to your dataset
        # Update 'your-dataset-name' to match your actual dataset name
        DATASET_NAME = 'your-dataset-name'  # ← UPDATE THIS!
        DATA_PATH = f'/kaggle/input/{DATASET_NAME}/'
        
        print(f"\n📁 Data path set to: {DATA_PATH}")
        
        # List contents of the dataset
        if os.path.exists(DATA_PATH):
            print(f"\nContents of dataset:")
            contents = os.listdir(DATA_PATH)
            for item in contents[:10]:
                print(f"  - {item}")
            if len(contents) > 10:
                print(f"  ... and {len(contents) - 10} more items")
        else:
            print(f"\n⚠️ WARNING: Dataset path {DATA_PATH} does not exist!")
            print(f"Please update DATASET_NAME to match your dataset folder name")
            
    except Exception as e:
        print(f"  Error listing datasets: {e}")
else:
    # Running locally (for testing)
    print("Not running on Kaggle. Using current directory for local testing.")
    DATA_PATH = './'

print(f"\nCurrent working directory: {os.getcwd()}")

## 3. Configuration and Paths

In [ ]:
# Configuration
CONFIG = {
    # Paths - These will point to your Kaggle dataset
    # The DATA_PATH is set in the previous cell
    'taxonomy_path': os.path.join(DATA_PATH, 'preprocessed_taxonomy.json'),
    'train_data_folder': os.path.join(DATA_PATH, 'train_data/'),
    'test_data_folder': os.path.join(DATA_PATH, 'test_data/'),
    'checkpoint_dir': 'checkpoints/',     # Local to Kaggle session
    'output_dir': 'outputs/',             # Local to Kaggle session (will be in Output tab)
    
    # Model configuration
    'model_name': 'distilbert-base-uncased',  # Faster and lighter than SciBERT
    'max_length': 512,                    # Maximum sequence length
    'batch_size': 8,                      # Batch size (reduce if OOM)
    'gradient_accumulation_steps': 4,     # Effective batch size = batch_size * this
    'learning_rate': 2e-5,
    'num_epochs': 5,
    'warmup_ratio': 0.1,
    'max_grad_norm': 1.0,
    
    # Training settings
    'save_steps': 500,                    # Save checkpoint every N steps
    'eval_steps': 500,                    # Evaluate every N steps
    'logging_steps': 100,
    'early_stopping_patience': 3,
    
    # Hierarchical classification
    'num_levels': 7,                      # Fixed to 7 levels
}

# Create directories
os.makedirs(CONFIG['checkpoint_dir'], exist_ok=True)
os.makedirs(CONFIG['output_dir'], exist_ok=True)

# Verify paths exist
print("Configuration loaded!")
print(f"Model: {CONFIG['model_name']}")
print(f"Effective batch size: {CONFIG['batch_size'] * CONFIG['gradient_accumulation_steps']}")
print(f"\n📂 Verifying data paths:")
print(f"  Taxonomy: {CONFIG['taxonomy_path']}")
print(f"    Exists: {os.path.exists(CONFIG['taxonomy_path'])}")
print(f"  Training data: {CONFIG['train_data_folder']}")
print(f"    Exists: {os.path.exists(CONFIG['train_data_folder'])}")
print(f"  Test data: {CONFIG['test_data_folder']}")
print(f"    Exists: {os.path.exists(CONFIG['test_data_folder'])}")

if not all([
    os.path.exists(CONFIG['taxonomy_path']),
    os.path.exists(CONFIG['train_data_folder']),
    os.path.exists(CONFIG['test_data_folder'])
]):
    print("\n⚠️ WARNING: Some data paths don't exist!")
    print("Please check:")
    print("  1. Your dataset is added to this Kaggle notebook (Add Data button)")
    print("  2. DATASET_NAME in the previous cell matches your dataset folder name")
    print("  3. Your dataset contains the required files/folders")
else:
    print("\n✓ All data paths verified!")

## 3. Load and Process Taxonomy

In [ ]:
def extract_all_paths(taxonomy_dict, current_path=[]):
    """
    Recursively extract all paths from the taxonomy.
    Returns a list of tuples representing hierarchical paths.
    """
    all_paths = []
    
    if isinstance(taxonomy_dict, dict):
        for key, value in taxonomy_dict.items():
            new_path = current_path + [key]
            if isinstance(value, dict):
                # Continue recursion
                all_paths.extend(extract_all_paths(value, new_path))
            elif isinstance(value, list):
                # Terminal level with list of items
                for item in value:
                    all_paths.append(tuple(new_path + [item]))
            else:
                # Single terminal node
                all_paths.append(tuple(new_path))
    elif isinstance(taxonomy_dict, list):
        # List of terminal nodes
        for item in taxonomy_dict:
            all_paths.append(tuple(current_path + [item]))
    
    return all_paths

def pad_path_to_7_levels(path):
    """
    Pad a path to exactly 7 levels by repeating the last element.
    Example: (A, B, C) -> (A, B, C, C, C, C, C)
    """
    path_list = list(path)
    while len(path_list) < 7:
        path_list.append(path_list[-1])
    return tuple(path_list[:7])  # Ensure exactly 7 levels

# Load taxonomy
print("Loading taxonomy...")
with open(CONFIG['taxonomy_path'], 'r', encoding='utf-8') as f:
    taxonomy_data = json.load(f)

# Extract all paths
all_paths = extract_all_paths(taxonomy_data['taxonomy'])
print(f"Total paths in taxonomy: {len(all_paths)}")

# Show example paths before padding
print("\nExample paths (before padding):")
for path in all_paths[:5]:
    print(f"  Length {len(path)}: {' > '.join(path)}")

# Pad all paths to 7 levels
padded_paths = [pad_path_to_7_levels(path) for path in all_paths]

# Show example paths after padding
print("\nExample paths (after padding to 7 levels):")
for i, path in enumerate(padded_paths[:5]):
    print(f"  {' > '.join(path)}")

# Create mapping of valid paths
valid_paths_set = set(padded_paths)
print(f"\nTotal unique 7-level paths: {len(valid_paths_set)}")

## 4. Create Label Encodings for Each Level

In [ ]:
# Create label encoders for each of the 7 levels
label_encoders = []
for level in range(7):
    # Get all unique labels at this level
    labels_at_level = set([path[level] for path in padded_paths])
    labels_sorted = sorted(list(labels_at_level))
    
    # Create mappings
    label_to_id = {label: idx for idx, label in enumerate(labels_sorted)}
    id_to_label = {idx: label for label, idx in label_to_id.items()}
    
    label_encoders.append({
        'label_to_id': label_to_id,
        'id_to_label': id_to_label,
        'num_labels': len(labels_sorted)
    })
    
    print(f"Level {level}: {len(labels_sorted)} unique labels")

# Display example encodings
print("\nExample label encodings for Level 0:")
for label, idx in list(label_encoders[0]['label_to_id'].items())[:10]:
    print(f"  {label} -> {idx}")

## 4.5. Build Parent-Child Taxonomy Mappings for Hierarchical Prediction

In [ ]:
def build_parent_child_mappings(padded_paths, label_encoders):
    """
    Build parent-child mappings for hierarchical prediction.
    For each level, map (parent_label) -> [valid_child_labels]
    
    Returns:
        List of dictionaries, one per level (starting from level 1)
        Each dict maps parent_label -> list of valid child label IDs
    """
    parent_child_maps = []
    
    # For each level (starting from level 1, since level 0 has no parent)
    for level in range(1, 7):
        parent_to_children = defaultdict(set)
        
        # Go through all paths and collect parent-child relationships
        for path in padded_paths:
            parent_label = path[level - 1]
            child_label = path[level]
            parent_to_children[parent_label].add(child_label)
        
        # Convert to label IDs and sorted lists
        parent_to_child_ids = {}
        for parent_label, child_labels in parent_to_children.items():
            # Get parent ID
            parent_id = label_encoders[level - 1]['label_to_id'][parent_label]
            
            # Get child IDs and sort them
            child_ids = sorted([
                label_encoders[level]['label_to_id'][child_label] 
                for child_label in child_labels
            ])
            
            parent_to_child_ids[parent_id] = child_ids
        
        parent_child_maps.append(parent_to_child_ids)
        
        print(f"Level {level}: Built mapping with {len(parent_to_child_ids)} parent categories")
    
    return parent_child_maps

# Build the mappings
print("Building parent-child taxonomy mappings for hierarchical prediction...")
parent_child_maps = build_parent_child_mappings(padded_paths, label_encoders)

# Display example mappings
print("\nExample: Level 1 children for first few Level 0 parents:")
for parent_id in list(parent_child_maps[0].keys())[:3]:
    parent_label = label_encoders[0]['id_to_label'][parent_id]
    child_ids = parent_child_maps[0][parent_id]
    child_labels = [label_encoders[1]['id_to_label'][cid] for cid in child_ids[:5]]
    print(f"  '{parent_label}' -> {child_labels}" + (" ..." if len(child_ids) > 5 else ""))

## 5. Load and Explore Data

In [ ]:
def load_json_files_from_folder(folder_path, sample_ratio=0.5):
    """Load JSON files from a folder and take only sample_ratio (50%) of data from each file."""
    data = []
    json_files = list(Path(folder_path).glob('*.json'))
    
    print(f"Found {len(json_files)} JSON files in {folder_path}")
    print(f"Sampling {sample_ratio*100}% of data from each file")
    
    for json_file in json_files:
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                file_data = json.load(f)
                # Handle both list and single object formats
                if isinstance(file_data, list):
                    # Take only sample_ratio of the data
                    sample_size = int(len(file_data) * sample_ratio)
                    sampled_data = file_data[:sample_size]
                    data.extend(sampled_data)
                    print(f"  {json_file.name}: {len(sampled_data)}/{len(file_data)} samples")
                else:
                    data.append(file_data)
        except Exception as e:
            print(f"Error loading {json_file}: {e}")
    
    return data

# Load training and test data
print("Loading training data...")
train_data = load_json_files_from_folder(CONFIG['train_data_folder'])
print(f"Total training samples: {len(train_data)}")

print("\nLoading test data...")
test_data = load_json_files_from_folder(CONFIG['test_data_folder'])
print(f"Total test samples: {len(test_data)}")

# Display a sample
sample = train_data[0]
print("\nSample training record:")
print(json.dumps(sample, indent=2)[:500] + "...")

## 6. Data Preprocessing

In [ ]:
def clean_text(text):
    """Clean and normalize text."""
    if not text:
        return ""
    
    # Remove HTML tags if any
    text = re.sub(r'<[^>]+>', '', str(text))
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)
    
    # Strip leading/trailing whitespace
    text = text.strip()
    
    return text

def parse_classification_path(path_string):
    """
    Parse classification path string into a list of labels.
    Example: "Natural Science > Mathematics > Algebra" -> ["Natural Science", "Mathematics", "Algebra"]
    """
    if not path_string:
        return None
    
    # Split by '>' and clean each part
    labels = [label.strip() for label in path_string.split('>')]
    
    # Remove empty labels
    labels = [label for label in labels if label]
    
    return labels if labels else None

def preprocess_sample(sample):
    """Preprocess a single data sample."""
    processed = {}
    
    # Extract title and abstract
    title = clean_text(sample.get('title', ''))
    abstract = clean_text(sample.get('abstract', ''))
    
    # Combine title and abstract
    processed['text'] = f"{title} [SEP] {abstract}"
    
    # Parse classification path
    classification_path = sample.get('classification_path', '')
    labels = parse_classification_path(classification_path)
    
    if not labels:
        return None
    
    # Pad labels to 7 levels
    padded_labels = labels.copy()
    while len(padded_labels) < 7:
        padded_labels.append(padded_labels[-1])
    padded_labels = padded_labels[:7]
    
    processed['labels'] = padded_labels
    processed['original_path'] = classification_path
    
    return processed

# Preprocess training data
print("Preprocessing training data...")
train_processed = []
for sample in train_data:
    processed = preprocess_sample(sample)
    if processed:
        train_processed.append(processed)

print(f"Successfully preprocessed {len(train_processed)}/{len(train_data)} training samples")

# Preprocess test data
print("\nPreprocessing test data...")
test_processed = []
for sample in test_data:
    processed = preprocess_sample(sample)
    if processed:
        test_processed.append(processed)

print(f"Successfully preprocessed {len(test_processed)}/{len(test_data)} test samples")

# Display example
print("\nExample preprocessed sample:")
example = train_processed[0]
print(f"Text (first 200 chars): {example['text'][:200]}...")
print(f"Labels: {' > '.join(example['labels'])}")

## 7. Create PyTorch Dataset and DataLoader

In [ ]:
# Initialize tokenizer
print(f"Loading tokenizer: {CONFIG['model_name']}")
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])

class HierarchicalDataset(Dataset):
    """Dataset for hierarchical text classification."""
    
    def __init__(self, data, tokenizer, label_encoders, max_length=512):
        self.data = data
        self.tokenizer = tokenizer
        self.label_encoders = label_encoders
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Tokenize text
        encoding = self.tokenizer(
            item['text'],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        # Convert labels to ids for each level
        label_ids = []
        for level, label in enumerate(item['labels']):
            label_id = self.label_encoders[level]['label_to_id'].get(label, 0)
            label_ids.append(label_id)
        
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(label_ids, dtype=torch.long)
        }

# Create datasets
train_dataset = HierarchicalDataset(
    train_processed,
    tokenizer,
    label_encoders,
    CONFIG['max_length']
)

test_dataset = HierarchicalDataset(
    test_processed,
    tokenizer,
    label_encoders,
    CONFIG['max_length']
)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=0  # Set to 0 for Kaggle
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=0
)

print(f"Training batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

# Test a batch
sample_batch = next(iter(train_loader))
print(f"\nSample batch shapes:")
print(f"  input_ids: {sample_batch['input_ids'].shape}")
print(f"  attention_mask: {sample_batch['attention_mask'].shape}")
print(f"  labels: {sample_batch['labels'].shape}")

## 8. Define Hierarchical Classification Model

In [ ]:
class HierarchicalClassifier(nn.Module):
    """
    Hierarchical text classifier with 7 classification heads.
    Predicts level-by-level, ensuring valid taxonomy paths.
    Uses LoRA (Low-Rank Adaptation) for parameter-efficient fine-tuning.
    """
    
    def __init__(self, model_name, label_encoders, parent_child_maps, use_lora=True):
        super(HierarchicalClassifier, self).__init__()
        
        # Load pretrained transformer
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size
        
        # Apply LoRA to the backbone for efficient fine-tuning
        if use_lora:
            lora_config = LoraConfig(
                r=8,                          # Rank of LoRA matrices (lower = fewer params)
                lora_alpha=16,                # Scaling factor
                target_modules=["q_lin", "v_lin"],  # For DistilBERT attention layers
                lora_dropout=0.1,
                bias="none",
                task_type=TaskType.FEATURE_EXTRACTION
            )
            self.backbone = get_peft_model(self.backbone, lora_config)
            print("LoRA applied to backbone:")
            self.backbone.print_trainable_parameters()
        
        # Create classification heads for each level (these are fully trained)
        self.classifiers = nn.ModuleList([
            nn.Linear(hidden_size, encoder['num_labels'])
            for encoder in label_encoders
        ])
        
        # Dropout for regularization
        self.dropout = nn.Dropout(0.1)
        
        # Store parent-child mappings and label encoders
        self.parent_child_maps = parent_child_maps
        self.label_encoders = label_encoders
        
    def forward(self, input_ids, attention_mask, hierarchical=False):
        """
        Forward pass with optional hierarchical prediction.
        
        Args:
            input_ids: Input token IDs
            attention_mask: Attention mask
            hierarchical: If True, predict sequentially ensuring valid paths.
                         If False, predict all levels independently (for training).
        
        Returns:
            List of logits for each level (7 tensors)
        """
        # Get transformer outputs
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        # Use [CLS] token representation
        pooled_output = outputs.last_hidden_state[:, 0, :]
        pooled_output = self.dropout(pooled_output)
        
        if not hierarchical:
            # Independent prediction for all levels (training mode)
            logits = []
            for classifier in self.classifiers:
                logits.append(classifier(pooled_output))
            return logits
        else:
            # Hierarchical prediction (inference mode)
            # Predict level-by-level, masking invalid children
            batch_size = input_ids.size(0)
            logits = []
            predictions = []
            
            # Level 0: No constraints
            level_0_logits = self.classifiers[0](pooled_output)
            logits.append(level_0_logits)
            level_0_preds = torch.argmax(level_0_logits, dim=-1)
            predictions.append(level_0_preds)
            
            # Levels 1-6: Constrained by parent
            for level in range(1, 7):
                level_logits = self.classifiers[level](pooled_output)
                
                # Mask invalid children based on parent prediction
                masked_logits = level_logits.clone()
                parent_preds = predictions[level - 1].cpu().numpy()
                
                # For each sample in batch
                for i in range(batch_size):
                    parent_id = int(parent_preds[i])
                    valid_child_ids = self.parent_child_maps[level - 1].get(parent_id, [])
                    
                    if valid_child_ids:
                        # Mask all invalid children with very negative value
                        mask = torch.ones(level_logits.size(1), device=level_logits.device) * float('-inf')
                        mask[valid_child_ids] = 0
                        masked_logits[i] = masked_logits[i] + mask
                
                logits.append(masked_logits)
                level_preds = torch.argmax(masked_logits, dim=-1)
                predictions.append(level_preds)
            
            return logits

# Initialize model with LoRA and parent-child mappings
print("Initializing hierarchical model with LoRA...")
model = HierarchicalClassifier(
    CONFIG['model_name'], 
    label_encoders, 
    parent_child_maps,
    use_lora=True
)
model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable %: {100 * trainable_params / total_params:.2f}%")

# Test forward pass (non-hierarchical for training)
with torch.no_grad():
    test_batch = next(iter(train_loader))
    test_input_ids = test_batch['input_ids'].to(device)
    test_attention_mask = test_batch['attention_mask'].to(device)
    test_logits = model(test_input_ids, test_attention_mask, hierarchical=False)
    print(f"\nForward pass (training mode) successful!")
    print(f"Number of classification heads: {len(test_logits)}")
    for i, logit in enumerate(test_logits):
        print(f"  Level {i} output shape: {logit.shape}")

# Test hierarchical forward pass
with torch.no_grad():
    test_logits_hier = model(test_input_ids, test_attention_mask, hierarchical=True)
    print(f"\nForward pass (hierarchical mode) successful!")
    print(f"Predictions will follow valid taxonomy paths!")

## 9. Training Setup with Checkpointing

In [ ]:
# Loss function - CrossEntropy for each level
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=0.01)

# Learning rate scheduler
num_training_steps = len(train_loader) * CONFIG['num_epochs'] // CONFIG['gradient_accumulation_steps']
num_warmup_steps = int(num_training_steps * CONFIG['warmup_ratio'])

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

print(f"Training steps: {num_training_steps}")
print(f"Warmup steps: {num_warmup_steps}")

# Checkpoint utilities
def save_checkpoint(model, optimizer, scheduler, epoch, step, best_f1, checkpoint_dir):
    """Save model checkpoint."""
    checkpoint = {
        'epoch': epoch,
        'step': step,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_f1': best_f1,
        'label_encoders': label_encoders
    }
    
    checkpoint_path = os.path.join(checkpoint_dir, f'checkpoint_epoch{epoch}_step{step}.pt')
    torch.save(checkpoint, checkpoint_path)
    print(f"Checkpoint saved: {checkpoint_path}")
    
    return checkpoint_path

def load_latest_checkpoint(checkpoint_dir, model, optimizer, scheduler):
    """Load the latest checkpoint if it exists."""
    checkpoints = list(Path(checkpoint_dir).glob('checkpoint_*.pt'))
    
    if not checkpoints:
        print("No checkpoint found. Starting from scratch.")
        return 0, 0, 0.0
    
    # Get the latest checkpoint
    latest_checkpoint = max(checkpoints, key=os.path.getctime)
    print(f"Loading checkpoint: {latest_checkpoint}")
    
    checkpoint = torch.load(latest_checkpoint, map_location=device)
    
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    
    print(f"Resumed from epoch {checkpoint['epoch']}, step {checkpoint['step']}")
    
    return checkpoint['epoch'], checkpoint['step'], checkpoint['best_f1']

# Try to load existing checkpoint
start_epoch, start_step, best_f1 = load_latest_checkpoint(
    CONFIG['checkpoint_dir'], model, optimizer, scheduler
)

print(f"\nTraining will start from:")
print(f"  Epoch: {start_epoch}")
print(f"  Step: {start_step}")
print(f"  Best F1: {best_f1:.4f}")

## 10. Training Loop

In [ ]:
from tqdm.auto import tqdm

# Training history
history = {
    'train_loss': [],
    'eval_f1': [],
    'level_f1': {i: [] for i in range(7)}
}

def train_epoch(model, train_loader, optimizer, scheduler, epoch):
    """Train for one epoch using non-hierarchical mode (independent predictions)."""
    model.train()
    total_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch}")
    
    optimizer.zero_grad()
    
    for step, batch in enumerate(progress_bar):
        # Move to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)  # Shape: [batch_size, 7]
        
        # Forward pass (non-hierarchical for training - learns all levels independently)
        logits = model(input_ids, attention_mask, hierarchical=False)
        
        # Calculate loss for each level
        loss = 0
        for level in range(7):
            level_loss = criterion(logits[level], labels[:, level])
            loss += level_loss
        
        # Average loss across levels
        loss = loss / 7
        
        # Backward pass with gradient accumulation
        loss = loss / CONFIG['gradient_accumulation_steps']
        loss.backward()
        
        if (step + 1) % CONFIG['gradient_accumulation_steps'] == 0:
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['max_grad_norm'])
            
            # Optimizer step
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        
        total_loss += loss.item() * CONFIG['gradient_accumulation_steps']
        progress_bar.set_postfix({'loss': loss.item() * CONFIG['gradient_accumulation_steps']})
    
    return total_loss / len(train_loader)

@torch.no_grad()
def evaluate(model, test_loader):
    """Evaluate model on test set using hierarchical prediction."""
    model.eval()
    
    all_predictions = {i: [] for i in range(7)}
    all_labels = {i: [] for i in range(7)}
    
    for batch in tqdm(test_loader, desc="Evaluating"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels']
        
        # Get predictions using hierarchical mode (ensures valid taxonomy paths)
        logits = model(input_ids, attention_mask, hierarchical=True)
        
        for level in range(7):
            preds = torch.argmax(logits[level], dim=-1).cpu().numpy()
            all_predictions[level].extend(preds)
            all_labels[level].extend(labels[:, level].numpy())
    
    # Calculate F1 for each level
    f1_scores = {}
    for level in range(7):
        f1 = f1_score(all_labels[level], all_predictions[level], average='macro', zero_division=0)
        f1_scores[level] = f1
    
    # Average F1 across all levels
    avg_f1 = np.mean(list(f1_scores.values()))
    
    return avg_f1, f1_scores, all_predictions, all_labels

print("Starting training...")
print("="*50)
print("Note: Training uses independent predictions, evaluation uses hierarchical prediction!")
print("This ensures all predictions follow valid taxonomy paths during inference.")

In [ ]:
# Main training loop
global_step = start_step
patience_counter = 0

for epoch in range(start_epoch, CONFIG['num_epochs']):
    print(f"\nEpoch {epoch + 1}/{CONFIG['num_epochs']}")
    
    # Train
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, epoch + 1)
    history['train_loss'].append(train_loss)
    print(f"Average training loss: {train_loss:.4f}")
    
    # Evaluate
    print("\nEvaluating on test set...")
    avg_f1, f1_scores, predictions, true_labels = evaluate(model, test_loader)
    history['eval_f1'].append(avg_f1)
    
    for level, f1 in f1_scores.items():
        history['level_f1'][level].append(f1)
    
    print(f"Average F1 Score: {avg_f1:.4f}")
    print("F1 Scores by Level:")
    for level, f1 in f1_scores.items():
        print(f"  Level {level}: {f1:.4f}")
    
    # Save checkpoint
    checkpoint_path = save_checkpoint(
        model, optimizer, scheduler, epoch + 1, global_step, best_f1, CONFIG['checkpoint_dir']
    )
    
    # Check for improvement
    if avg_f1 > best_f1:
        best_f1 = avg_f1
        patience_counter = 0
        
        # Save best model
        best_model_path = os.path.join(CONFIG['output_dir'], 'best_model.pt')
        torch.save({
            'model_state_dict': model.state_dict(),
            'label_encoders': label_encoders,
            'f1_score': best_f1
        }, best_model_path)
        print(f"New best F1: {best_f1:.4f}! Model saved.")
    else:
        patience_counter += 1
        print(f"No improvement. Patience: {patience_counter}/{CONFIG['early_stopping_patience']}")
        
        if patience_counter >= CONFIG['early_stopping_patience']:
            print("\nEarly stopping triggered!")
            break
    
    global_step += len(train_loader)

print("\n" + "="*50)
print("Training completed!")
print(f"Best F1 Score: {best_f1:.4f}")

## 10.5. Package Model for Download (Kaggle)

In [ ]:
import shutil
from datetime import datetime

# Create a downloadable package of all model artifacts
print("Packaging model for download...")

# Create a timestamped folder for this model
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
package_dir = f"hierarchical_model_{timestamp}"
os.makedirs(package_dir, exist_ok=True)

# Save the best model
print("\n1. Saving best model...")
if os.path.exists(os.path.join(CONFIG['output_dir'], 'best_model.pt')):
    shutil.copy(
        os.path.join(CONFIG['output_dir'], 'best_model.pt'),
        os.path.join(package_dir, 'best_model.pt')
    )
    print(f"   ✓ Best model saved to {package_dir}/best_model.pt")

# Save tokenizer
print("\n2. Saving tokenizer...")
tokenizer_path = os.path.join(package_dir, 'tokenizer')
tokenizer.save_pretrained(tokenizer_path)
print(f"   ✓ Tokenizer saved to {tokenizer_path}")

# Save label encoders
print("\n3. Saving label encoders...")
with open(os.path.join(package_dir, 'label_encoders.json'), 'w') as f:
    encoders_json = []
    for encoder in label_encoders:
        encoders_json.append({
            'label_to_id': encoder['label_to_id'],
            'id_to_label': {str(k): v for k, v in encoder['id_to_label'].items()},
            'num_labels': encoder['num_labels']
        })
    json.dump(encoders_json, f, indent=2)
print(f"   ✓ Label encoders saved")

# Save parent-child mappings
print("\n4. Saving parent-child mappings...")
with open(os.path.join(package_dir, 'parent_child_maps.json'), 'w') as f:
    maps_json = []
    for level_map in parent_child_maps:
        maps_json.append({
            str(parent_id): child_ids 
            for parent_id, child_ids in level_map.items()
        })
    json.dump(maps_json, f, indent=2)
print(f"   ✓ Parent-child mappings saved")

# Save training configuration
print("\n5. Saving configuration...")
with open(os.path.join(package_dir, 'config.json'), 'w') as f:
    json.dump(CONFIG, f, indent=2)
print(f"   ✓ Configuration saved")

# Save training metrics
print("\n6. Saving training metrics...")
metrics = {
    'best_f1': float(best_f1),
    'training_history': {
        'train_loss': [float(x) for x in history['train_loss']],
        'eval_f1': [float(x) for x in history['eval_f1']],
    }
}
with open(os.path.join(package_dir, 'training_metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"   ✓ Training metrics saved")

# Create a README with usage instructions
print("\n7. Creating README...")
readme_content = f"""# Hierarchical Text Classification Model

## Model Information
- **Training Date**: {timestamp}
- **Model Type**: {CONFIG['model_name']}
- **Best F1 Score**: {best_f1:.4f}
- **Taxonomy Levels**: 7

## Files Included

1. `best_model.pt` - Trained model weights
2. `tokenizer/` - Tokenizer files
3. `label_encoders.json` - Label mappings for all 7 levels
4. `parent_child_maps.json` - Taxonomy parent-child relationships (REQUIRED for hierarchical inference)
5. `config.json` - Training configuration
6. `training_metrics.json` - Training history and metrics

## How to Use

```python
import torch
import json
from transformers import AutoTokenizer

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained('tokenizer/')

# Load label encoders
with open('label_encoders.json', 'r') as f:
    encoders_json = json.load(f)
    label_encoders = []
    for encoder in encoders_json:
        label_encoders.append({{
            'label_to_id': encoder['label_to_id'],
            'id_to_label': {{int(k): v for k, v in encoder['id_to_label'].items()}},
            'num_labels': encoder['num_labels']
        }})

# Load parent-child mappings
with open('parent_child_maps.json', 'r') as f:
    maps_json = json.load(f)
    parent_child_maps = []
    for level_map in maps_json:
        parent_child_maps.append({{
            int(parent_id): child_ids 
            for parent_id, child_ids in level_map.items()
        }})

# Load model
from your_model_file import HierarchicalClassifier  # You need to copy the model class

model = HierarchicalClassifier(
    '{CONFIG['model_name']}',
    label_encoders,
    parent_child_maps,
    use_lora=True
)

checkpoint = torch.load('best_model.pt', map_location='cpu')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Make predictions (hierarchical mode - ensures valid taxonomy paths)
def predict(title, abstract):
    text = f"{{title}} [SEP] {{abstract}}"
    encoding = tokenizer(text, max_length=512, padding='max_length', truncation=True, return_tensors='pt')
    
    with torch.no_grad():
        logits = model(encoding['input_ids'], encoding['attention_mask'], hierarchical=True)
    
    predicted_path = []
    for level in range(7):
        pred_id = torch.argmax(logits[level], dim=-1).item()
        pred_label = label_encoders[level]['id_to_label'][pred_id]
        predicted_path.append(pred_label)
    
    return ' > '.join(predicted_path)

# Example
result = predict("Your paper title", "Your paper abstract")
print(result)
```

## Important Notes

⚠️ **Required Files**: All files in this package are required for inference. The `parent_child_maps.json` is especially critical for hierarchical prediction.

✓ **Valid Paths**: This model uses hierarchical prediction, guaranteeing 100% valid taxonomy paths.

## Model Class

You need to copy the `HierarchicalClassifier` class definition from your training notebook to use this model.
"""

with open(os.path.join(package_dir, 'README.md'), 'w') as f:
    f.write(readme_content)
print(f"   ✓ README created")

# Create a zip file for easy download
print("\n8. Creating zip file...")
zip_filename = f"{package_dir}.zip"
shutil.make_archive(package_dir, 'zip', package_dir)
print(f"   ✓ Zip file created: {zip_filename}")

# Display summary
print("\n" + "="*60)
print("MODEL PACKAGE READY FOR DOWNLOAD!")
print("="*60)
print(f"\nPackage folder: {package_dir}/")
print(f"Zip file: {zip_filename}")
print(f"\nTotal size: {sum(os.path.getsize(os.path.join(dirpath, filename)) for dirpath, dirnames, filenames in os.walk(package_dir) for filename in filenames) / (1024*1024):.2f} MB")

print("\n📥 How to Download from Kaggle:")
print("   1. Wait for the notebook to finish running")
print("   2. Click 'Output' tab on the right side")
print(f"   3. Download '{zip_filename}' file")
print("   4. Extract the zip file on your local machine")
print("   5. Follow instructions in README.md to use the model")

print("\n✓ All artifacts are also saved in the 'outputs/' directory")
print("✓ You can access them in future Kaggle sessions by enabling 'Output' as input")

# List all files in the package
print(f"\n📦 Package contents:")
for root, dirs, files in os.walk(package_dir):
    level = root.replace(package_dir, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        file_size = os.path.getsize(os.path.join(root, file)) / 1024
        print(f'{subindent}{file} ({file_size:.1f} KB)')

## 11. Load Best Model and Final Evaluation

In [ ]:
# Load best model
best_model_path = os.path.join(CONFIG['output_dir'], 'best_model.pt')
if os.path.exists(best_model_path):
    print(f"Loading best model from {best_model_path}")
    checkpoint = torch.load(best_model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Best model F1 score: {checkpoint['f1_score']:.4f}")
else:
    print("No best model found. Using current model.")

# Final evaluation
print("\nFinal evaluation on test set...")
final_avg_f1, final_f1_scores, final_predictions, final_true_labels = evaluate(model, test_loader)

print(f"\nFinal Results:")
print(f"Average F1 Score: {final_avg_f1:.4f}")
print("\nF1 Scores by Level:")
for level, f1 in final_f1_scores.items():
    print(f"  Level {level}: {f1:.4f}")

## 12. Detailed Classification Reports

In [ ]:
# Generate classification reports for each level
print("Classification Reports:\n")

for level in range(7):
    print(f"\n{'='*60}")
    print(f"Level {level} Classification Report")
    print('='*60)
    
    # Get label names
    label_names = [label_encoders[level]['id_to_label'][i] 
                   for i in range(len(label_encoders[level]['id_to_label']))]
    
    # Calculate metrics
    y_true = final_true_labels[level]
    y_pred = final_predictions[level]
    
    # Print report
    report = classification_report(
        y_true, y_pred,
        target_names=label_names[:min(50, len(label_names))],  # Limit to 50 for readability
        zero_division=0
    )
    print(report)
    
    # Calculate accuracy
    acc = accuracy_score(y_true, y_pred)
    print(f"Accuracy: {acc:.4f}")

## 13. Visualization

In [ ]:
# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Create figure with multiple subplots
fig = plt.figure(figsize=(20, 12))

# 1. Training Loss Curve
ax1 = plt.subplot(2, 3, 1)
epochs = range(1, len(history['train_loss']) + 1)
ax1.plot(epochs, history['train_loss'], 'b-o', linewidth=2, markersize=8)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Training Loss', fontsize=12)
ax1.set_title('Training Loss Over Epochs', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

# 2. F1 Score Over Epochs
ax2 = plt.subplot(2, 3, 2)
ax2.plot(epochs, history['eval_f1'], 'g-o', linewidth=2, markersize=8)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Macro F1 Score', fontsize=12)
ax2.set_title('Average F1 Score Over Epochs', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

# 3. F1 Score by Level (Bar Plot)
ax3 = plt.subplot(2, 3, 3)
levels = list(range(7))
final_f1_values = [final_f1_scores[i] for i in levels]
bars = ax3.bar(levels, final_f1_values, color=sns.color_palette("viridis", 7), edgecolor='black')
ax3.set_xlabel('Hierarchy Level', fontsize=12)
ax3.set_ylabel('Macro F1 Score', fontsize=12)
ax3.set_title('F1 Score by Hierarchy Level', fontsize=14, fontweight='bold')
ax3.set_xticks(levels)
ax3.set_ylim([0, 1])
for i, (bar, val) in enumerate(zip(bars, final_f1_values)):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{val:.3f}', ha='center', va='bottom', fontsize=10)
ax3.grid(True, alpha=0.3, axis='y')

# 4. Accuracy by Level
ax4 = plt.subplot(2, 3, 4)
accuracies = [accuracy_score(final_true_labels[i], final_predictions[i]) for i in range(7)]
bars = ax4.bar(levels, accuracies, color=sns.color_palette("mako", 7), edgecolor='black')
ax4.set_xlabel('Hierarchy Level', fontsize=12)
ax4.set_ylabel('Accuracy', fontsize=12)
ax4.set_title('Accuracy by Hierarchy Level', fontsize=14, fontweight='bold')
ax4.set_xticks(levels)
ax4.set_ylim([0, 1])
for i, (bar, val) in enumerate(zip(bars, accuracies)):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{val:.3f}', ha='center', va='bottom', fontsize=10)
ax4.grid(True, alpha=0.3, axis='y')

# 5. Level-wise F1 Progression
ax5 = plt.subplot(2, 3, 5)
for level in range(7):
    if history['level_f1'][level]:
        ax5.plot(range(1, len(history['level_f1'][level]) + 1), 
                history['level_f1'][level], 
                marker='o', label=f'Level {level}', linewidth=2)
ax5.set_xlabel('Epoch', fontsize=12)
ax5.set_ylabel('F1 Score', fontsize=12)
ax5.set_title('F1 Score Progression by Level', fontsize=14, fontweight='bold')
ax5.legend(loc='best', fontsize=10)
ax5.grid(True, alpha=0.3)

# 6. Confusion Matrix for Level 0 (sample)
ax6 = plt.subplot(2, 3, 6)
cm = confusion_matrix(final_true_labels[0], final_predictions[0])
# Normalize for better visualization
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_normalized[:10, :10], annot=False, fmt='.2f', cmap='YlOrRd', 
            ax=ax6, cbar_kws={'label': 'Proportion'})
ax6.set_title('Confusion Matrix (Level 0, Top 10 Classes)', fontsize=14, fontweight='bold')
ax6.set_xlabel('Predicted', fontsize=12)
ax6.set_ylabel('True', fontsize=12)

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'training_results.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"Visualization saved to {CONFIG['output_dir']}/training_results.png")

## 14. Hierarchical Metrics

In [ ]:
def calculate_hierarchical_metrics(true_labels, predictions, label_encoders):
    """
    Calculate hierarchical precision, recall, and F1.
    Considers hierarchical consistency - predictions should follow valid taxonomy paths.
    """
    # Convert predictions and labels to paths
    num_samples = len(true_labels[0])
    
    # Calculate hierarchical accuracy (correct at all levels up to k)
    hierarchical_accuracy = []
    for k in range(1, 8):
        correct = 0
        for i in range(num_samples):
            # Check if first k levels are all correct
            if all(predictions[level][i] == true_labels[level][i] for level in range(k)):
                correct += 1
        hierarchical_accuracy.append(correct / num_samples)
    
    # Calculate path consistency (whether predictions form valid taxonomy paths)
    valid_path_count = 0
    for i in range(num_samples):
        # Reconstruct predicted path
        pred_path = tuple([label_encoders[level]['id_to_label'][predictions[level][i]] 
                          for level in range(7)])
        
        # Check if path exists in valid paths
        if pred_path in valid_paths_set:
            valid_path_count += 1
    
    path_consistency = valid_path_count / num_samples
    
    return {
        'hierarchical_accuracy': hierarchical_accuracy,
        'path_consistency': path_consistency
    }

# Calculate hierarchical metrics
print("Calculating hierarchical metrics...")
hier_metrics = calculate_hierarchical_metrics(final_true_labels, final_predictions, label_encoders)

print("\nHierarchical Accuracy (correct up to level k):")
for k, acc in enumerate(hier_metrics['hierarchical_accuracy'], 1):
    print(f"  Up to Level {k}: {acc:.4f}")

print(f"\nPath Consistency (valid taxonomy paths): {hier_metrics['path_consistency']:.4f}")
print("✓ With hierarchical prediction, this should be 100% (all predictions are valid paths)!")

# Visualize hierarchical accuracy
plt.figure(figsize=(10, 6))
levels_k = list(range(1, 8))
plt.plot(levels_k, hier_metrics['hierarchical_accuracy'], 'b-o', linewidth=2, markersize=10)
plt.xlabel('Hierarchy Depth (up to level k)', fontsize=12)
plt.ylabel('Hierarchical Accuracy', fontsize=12)
plt.title('Hierarchical Accuracy: Correct Predictions up to Level k', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.xticks(levels_k)
plt.ylim([0, 1])
for i, acc in enumerate(hier_metrics['hierarchical_accuracy']):
    plt.text(i+1, acc + 0.02, f'{acc:.3f}', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'hierarchical_accuracy.png'), dpi=300, bbox_inches='tight')
plt.show()

## 15. Save Final Model and Metadata

In [ ]:
# Save tokenizer
tokenizer.save_pretrained(os.path.join(CONFIG['output_dir'], 'tokenizer'))
print(f"Tokenizer saved to {CONFIG['output_dir']}/tokenizer")

# Save label encoders
with open(os.path.join(CONFIG['output_dir'], 'label_encoders.json'), 'w') as f:
    # Convert to JSON-serializable format
    encoders_json = []
    for encoder in label_encoders:
        encoders_json.append({
            'label_to_id': encoder['label_to_id'],
            'id_to_label': {str(k): v for k, v in encoder['id_to_label'].items()},
            'num_labels': encoder['num_labels']
        })
    json.dump(encoders_json, f, indent=2)
print(f"Label encoders saved to {CONFIG['output_dir']}/label_encoders.json")

# Save parent-child mappings (critical for hierarchical prediction!)
with open(os.path.join(CONFIG['output_dir'], 'parent_child_maps.json'), 'w') as f:
    # Convert to JSON-serializable format (keys must be strings)
    maps_json = []
    for level_map in parent_child_maps:
        maps_json.append({
            str(parent_id): child_ids 
            for parent_id, child_ids in level_map.items()
        })
    json.dump(maps_json, f, indent=2)
print(f"Parent-child mappings saved to {CONFIG['output_dir']}/parent_child_maps.json")
print("  ⚠️ These mappings are REQUIRED for hierarchical inference!")

# Save evaluation results
results = {
    'final_avg_f1': float(final_avg_f1),
    'f1_by_level': {f'level_{k}': float(v) for k, v in final_f1_scores.items()},
    'accuracies_by_level': {f'level_{k}': float(accuracy_score(final_true_labels[k], final_predictions[k])) 
                            for k in range(7)},
    'hierarchical_accuracy': [float(x) for x in hier_metrics['hierarchical_accuracy']],
    'path_consistency': float(hier_metrics['path_consistency']),
    'training_history': {
        'train_loss': [float(x) for x in history['train_loss']],
        'eval_f1': [float(x) for x in history['eval_f1']]
    },
    'config': CONFIG
}

with open(os.path.join(CONFIG['output_dir'], 'evaluation_results.json'), 'w') as f:
    json.dump(results, f, indent=2)
print(f"Evaluation results saved to {CONFIG['output_dir']}/evaluation_results.json")

print("\nAll artifacts saved successfully!")
print("✓ Model, tokenizer, encoders, and parent-child maps ready for deployment!")

## 16. Inference Function for New Papers

In [ ]:
def predict_classification(title, abstract, model, tokenizer, label_encoders, device='cuda'):
    """
    Predict hierarchical classification for a new research paper.
    Uses hierarchical mode to ensure predictions follow valid taxonomy paths.
    
    Args:
        title: Paper title
        abstract: Paper abstract
        model: Trained model
        tokenizer: Tokenizer
        label_encoders: Label encoders for each level
        device: Device to run inference on
    
    Returns:
        Dictionary with predicted labels for each level and full path
    """
    model.eval()
    
    # Prepare text
    text = f"{clean_text(title)} [SEP] {clean_text(abstract)}"
    
    # Tokenize
    encoding = tokenizer(
        text,
        max_length=CONFIG['max_length'],
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    
    # Move to device
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    # Predict using hierarchical mode (guarantees valid taxonomy path)
    with torch.no_grad():
        logits = model(input_ids, attention_mask, hierarchical=True)
    
    # Get predictions for each level
    predictions = {}
    predicted_path = []
    
    for level in range(7):
        pred_id = torch.argmax(logits[level], dim=-1).item()
        pred_label = label_encoders[level]['id_to_label'][pred_id]
        predictions[f'level_{level}'] = pred_label
        predicted_path.append(pred_label)
    
    predictions['full_path'] = ' > '.join(predicted_path)
    predictions['is_valid_path'] = True  # Always true with hierarchical prediction
    
    return predictions

# Example usage
print("Testing hierarchical inference on a sample paper...")
sample_paper = test_processed[0]

# Extract title and abstract from original text
text_parts = sample_paper['text'].split('[SEP]')
sample_title = text_parts[0].strip() if len(text_parts) > 0 else ""
sample_abstract = text_parts[1].strip() if len(text_parts) > 1 else ""

predicted = predict_classification(
    sample_title,
    sample_abstract,
    model,
    tokenizer,
    label_encoders,
    device
)

print("\nSample Paper:")
print(f"Title: {sample_title[:100]}...")
print(f"Abstract: {sample_abstract[:200]}...")
print(f"\n✓ Predicted Classification Path (Hierarchical):")
print(predicted['full_path'])
print(f"Valid Taxonomy Path: {predicted['is_valid_path']}")
print(f"\nActual Classification Path:")
print(sample_paper['original_path'])

# Show level-by-level breakdown
print("\nLevel-by-level Predictions:")
for level in range(7):
    pred = predicted[f'level_{level}']
    actual = sample_paper['labels'][level]
    match = "✓" if pred == actual else "✗"
    print(f"  Level {level}: {pred} {match} (actual: {actual})")

## 17. Summary and Next Steps

### Summary

This notebook implements a **hierarchical text classification pipeline** that predicts level-by-level, ensuring all predictions follow valid taxonomy paths:

#### Key Features:

1. **True Hierarchical Prediction**: 
   - Each level predicts based on the parent level's prediction
   - Guarantees 100% valid taxonomy paths (no impossible category combinations)
   - Uses parent-child mappings from your taxonomy structure

2. **Dual-Mode Architecture**:
   - **Training**: Independent predictions at all levels (learns broadly)
   - **Inference**: Sequential hierarchical predictions (enforces valid paths)

3. **Taxonomy Processing**: 
   - Automatically padded all classification paths to exactly 7 levels
   - Built parent-child mappings for hierarchical constraints

4. **Efficient Fine-tuning**: 
   - LoRA (Low-Rank Adaptation) for parameter-efficient training
   - Gradient accumulation for larger effective batch sizes
   - Memory-optimized for limited GPU resources

5. **Checkpoint System**: 
   - Automatic checkpoint saving and loading for session recovery
   - Early stopping to prevent overfitting

6. **Comprehensive Evaluation**: 
   - Level-wise F1 scores and accuracy
   - Hierarchical accuracy (correct up to level k)
   - Path consistency verification (should be 100%)

### How Hierarchical Prediction Works:

```
Level 0: Predict among ALL top-level categories
         ↓
Level 1: Predict among ONLY valid children of Level 0 prediction
         ↓
Level 2: Predict among ONLY valid children of Level 1 prediction
         ↓
  ... and so on for all 7 levels
```

This ensures predictions like "Natural Science > Computer Science" are impossible if "Computer Science" isn't a child of "Natural Science" in your taxonomy.

### Key Results

- **Average F1 Score**: Performance across all levels
- **Hierarchical Accuracy**: Accuracy when requiring ALL levels up to k to be correct
- **Path Consistency**: 100% (all predictions are valid taxonomy paths)
- **Level-wise Performance**: Individual metrics showing which levels are easier/harder

### Files Saved

- `checkpoints/`: Model checkpoints for session recovery
- `outputs/best_model.pt`: Best performing model with parent-child mappings
- `outputs/tokenizer/`: Tokenizer files
- `outputs/label_encoders.json`: Label encodings for all 7 levels
- `outputs/evaluation_results.json`: Complete evaluation metrics
- `outputs/training_results.png`: Visualization plots
- `outputs/hierarchical_accuracy.png`: Hierarchical accuracy plot

### Next Steps

1. **Error Analysis**: Examine which taxonomy paths are most confused
2. **Parent-Level Impact**: Analyze how errors propagate through the hierarchy
3. **Model Optimization**: Try SciBERT for scientific domain or other domain-specific models
4. **Data Augmentation**: Augment training data with paraphrasing or back-translation
5. **Deployment**: Package model for production use with FastAPI or similar framework

### Usage for New Predictions

```python
# Load the model and predict (hierarchical mode ensures valid paths)
from transformers import AutoTokenizer

# Load components
tokenizer = AutoTokenizer.from_pretrained('outputs/tokenizer')

# Rebuild model with parent-child maps
model = HierarchicalClassifier(
    CONFIG['model_name'], 
    label_encoders, 
    parent_child_maps,
    use_lora=True
)
checkpoint = torch.load('outputs/best_model.pt')
model.load_state_dict(checkpoint['model_state_dict'])
model.to('cuda')

# Predict (automatically uses hierarchical mode)
result = predict_classification(
    title="Your paper title",
    abstract="Your paper abstract",
    model=model,
    tokenizer=tokenizer,
    label_encoders=label_encoders
)
print(result['full_path'])  # Guaranteed to be a valid taxonomy path!
print(f"Valid: {result['is_valid_path']}")  # Always True
```

### Important Notes

⚠️ **Model Dependencies**: The model requires `parent_child_maps` which are built from your taxonomy. Make sure to save and load these mappings along with the model.

✓ **100% Path Validity**: With hierarchical prediction, ALL predictions will be valid paths from your taxonomy - no impossible category combinations!